In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df[["time", "X", "Y", "ndvi"]]

In [ ]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

In [2]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [ ]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=Plumas_Boundaries.geometry(), crs='EPSG:5070', scale=5000)

In [ ]:
print(ds)

In [ ]:
# 5. Preview the dataset and the results before doing a full run!
# This allows users to inspect the structure and content of the data to ensure it behaves as expected prior to running a full computation.
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
},
    user_function=compute_ndvi,
    preview_dataset=True
)

In [ ]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["time", "X", "Y", "ndvi"]
chunks = {"time": 1, "X": 1024, "Y": 512}

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:5070',
    'scale': 1000,
    'chunks': chunks
},
    user_function=compute_ndvi,
    output_template=df_list,
    user_function_kwargs={
    },
    export_kwargs={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "US_NDVI_Tiles_US_1000m",
        "vrt": True,
        #"chunks": chunks,
        "report": True}, 
    dask_mode="custom",
    dask_kwargs={
        "n_workers": 30,
        "threads_per_worker": 1,
        "memory_limit": "2GB"
    }
)

In [ ]:
# Docker version
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': Plumas_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
    },
    export_kwargs={
            "mode": "raster",
            "file_format": "GTiff",
            "output_folder": "tiles33"
    },
    dask_mode="custom",
    dask_kwargs={
            "n_workers": 4,
            "threads_per_worker": 1,
            "memory_limit": "1GB"
    },
    user_function=compute_ndvi,
    dask_use_docker=True,
    dask_docker_image="adrianomdocker/rr041"
)